In [ ]:
# ============================================================
# EXPERIMENT RUNNER — PRODUCTION VERSION
#
# Models:
#   LightGBM
#   Random Forest
#   XGBoost
#
# Experiment:
#   Fixed 12-case cohort
#   lambda = 0.0, 0.1, ..., 1.0
#   3 ORs
#   480-min nominal capacity
#   20-min turnover
#   beta = 0.10
#   360-min duration cap
#   300-s solver time limit
#
# Total planned runs:
#   3 models × 11 lambda values = 33 runs
# ============================================================

import os
import time
import numpy as np
import pandas as pd

# Load the final strengthened Big-M MILP engine.
%run milp_engine.ipynb

print("Final strengthened MILP engine loaded.")

In [ ]:
# ============================================================
# 1. FROZEN EXPERIMENT CONFIGURATION
# ============================================================

MODELS = {

    "LightGBM":
        "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",

    "RF":
        "Prescriptive/RF_prescriptive_optimizer_inputs.csv",

    "XGBoost":
        "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv"
}


LAMBDA_LIST = np.round(
    np.linspace(0.0, 1.0, 11),
    2
)


N_CASES = 12

N_ROOMS = 3

ROOM_CAPACITY = 480

TURNOVER = 20

BALANCE_WEIGHT = 0.10

DURATION_CAP = 360.0

TIME_LIMIT = 300


COHORT_PATH = "Cohort/fixed_cohort.csv"

RESULTS_DIR = "Results"

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "all_models_summary_checkpoint.csv"
)

MASTER_PATH = os.path.join(
    RESULTS_DIR,
    "all_models_summary.csv"
)

SOLVER_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "solver_performance_summary.csv"
)


os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


# ============================================================
# 2. PROTECT THE FROZEN COHORT
# ============================================================

assert os.path.exists(COHORT_PATH), (
    f"Frozen cohort not found: {COHORT_PATH}\n"
    "Production experiments must NOT automatically resample "
    "a new cohort."
)


cohort_ids = pd.read_csv(
    COHORT_PATH
)


# ------------------------------------------------------------
# Basic cohort integrity
# ------------------------------------------------------------

assert "LOG_ID" in cohort_ids.columns, (
    "fixed_cohort.csv does not contain LOG_ID."
)

assert len(cohort_ids) == N_CASES, (
    f"Expected {N_CASES} fixed cases, "
    f"found {len(cohort_ids)}."
)

assert cohort_ids["LOG_ID"].is_unique, (
    "Duplicate LOG_ID found in fixed cohort."
)

assert cohort_ids["LOG_ID"].notna().all(), (
    "Missing LOG_ID found in fixed cohort."
)


# Store as strings for robust cross-file comparison.

frozen_ids = (
    cohort_ids["LOG_ID"]
    .astype(str)
    .tolist()
)


print("=" * 70)
print("FROZEN EXPERIMENT CONFIGURATION")
print("=" * 70)

print(f"Models                 : {list(MODELS.keys())}")
print(f"Lambda values          : {LAMBDA_LIST.tolist()}")
print(f"Fixed cohort size      : {N_CASES}")
print(f"Rooms                  : {N_ROOMS}")
print(f"Room capacity          : {ROOM_CAPACITY} min")
print(f"Turnover               : {TURNOVER} min")
print(f"Balance weight (beta)  : {BALANCE_WEIGHT}")
print(f"Duration cap           : {DURATION_CAP} min")
print(f"Time limit             : {TIME_LIMIT} s")
print(f"Planned runs           : {len(MODELS) * len(LAMBDA_LIST)}")


# ============================================================
# 3. VERIFY ALL OPTIMIZER INPUTS BEFORE ANY MILP RUN
# ============================================================

verified_model_data = {}


for model_name, model_path in MODELS.items():

    print("\n" + "-" * 70)
    print(f"Verifying {model_name}")
    print("-" * 70)

    assert os.path.exists(model_path), (
        f"{model_name}: optimizer input not found:\n"
        f"{model_path}"
    )

    df = pd.read_csv(
        model_path
    )


    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required_columns = {
        "LOG_ID",
        "DURATION_P50_MINS",
        "DURATION_P90_MINS"
    }

    missing_columns = (
        required_columns
        -
        set(df.columns)
    )

    assert len(missing_columns) == 0, (
        f"{model_name}: missing required columns: "
        f"{missing_columns}"
    )


    # --------------------------------------------------------
    # LOG_ID integrity
    # --------------------------------------------------------

    assert df["LOG_ID"].notna().all(), (
        f"{model_name}: missing LOG_ID values."
    )

    assert df["LOG_ID"].is_unique, (
        f"{model_name}: duplicate LOG_ID values found."
    )


    # --------------------------------------------------------
    # Prediction integrity
    # --------------------------------------------------------

    assert df[
        [
            "DURATION_P50_MINS",
            "DURATION_P90_MINS"
        ]
    ].notna().all().all(), (
        f"{model_name}: missing duration predictions."
    )

    assert np.isfinite(
        df[
            [
                "DURATION_P50_MINS",
                "DURATION_P90_MINS"
            ]
        ].to_numpy(dtype=float)
    ).all(), (
        f"{model_name}: non-finite duration predictions found."
    )

    assert (
        df["DURATION_P50_MINS"] > 0
    ).all(), (
        f"{model_name}: non-positive P50 predictions found."
    )

    assert (
        df["DURATION_P90_MINS"] > 0
    ).all(), (
        f"{model_name}: non-positive P90 predictions found."
    )


    # P90 should not be below P50 for the interpolation used
    # by the prescriptive layer.

    assert (
        df["DURATION_P90_MINS"]
        >=
        df["DURATION_P50_MINS"]
    ).all(), (
        f"{model_name}: at least one P90 prediction is "
        f"below P50."
    )


    # --------------------------------------------------------
    # Fixed cohort compatibility
    # --------------------------------------------------------

    df_id_strings = (
        df["LOG_ID"]
        .astype(str)
    )

    missing_ids = (
        set(frozen_ids)
        -
        set(df_id_strings)
    )

    assert len(missing_ids) == 0, (
        f"{model_name}: missing frozen cohort IDs:\n"
        f"{sorted(missing_ids)}"
    )


    # --------------------------------------------------------
    # Extract in EXACT frozen cohort order
    # --------------------------------------------------------

    df_indexed = df.copy()

    df_indexed["LOG_ID"] = (
        df_indexed["LOG_ID"]
        .astype(str)
    )

    df_day = (
        df_indexed
        .set_index("LOG_ID")
        .loc[frozen_ids]
        .reset_index()
    )


    assert len(df_day) == N_CASES, (
        f"{model_name}: expected {N_CASES} cohort cases, "
        f"found {len(df_day)}."
    )


    assert (
        df_day["LOG_ID"].tolist()
        ==
        frozen_ids
    ), (
        f"{model_name}: frozen cohort order mismatch."
    )


    # --------------------------------------------------------
    # Save verified data in memory
    # --------------------------------------------------------

    verified_model_data[
        model_name
    ] = df_day


    print(
        f"{model_name}: PASS "
        f"| full input = {df.shape} "
        f"| fixed cohort = {df_day.shape}"
    )


# ============================================================
# 4. CROSS-MODEL COHORT ALIGNMENT
# ============================================================

reference_ids = (
    verified_model_data["LightGBM"]["LOG_ID"]
    .tolist()
)


for model_name in MODELS:

    model_ids = (
        verified_model_data[model_name]["LOG_ID"]
        .tolist()
    )

    assert model_ids == reference_ids, (
        f"Cross-model cohort alignment failed for "
        f"{model_name}."
    )


print("\n" + "=" * 70)
print("PRE-RUN INTEGRITY CHECK: PASS")
print("=" * 70)

print(
    "All three leakage-free optimizer inputs contain "
    "the identical frozen 12-case cohort in identical order."
)

print(
    "No MILP experiment has been started yet."
)

In [ ]:
# ============================================================
# PRODUCTION MILP EXPERIMENT
#
# 3 models × 11 lambda values = 33 runs
# ============================================================


# ------------------------------------------------------------
# Keys containing non-scalar schedule details.
#
# These are useful for schedule inspection but should NOT be
# written into the scalar summary CSV.
# ------------------------------------------------------------

NON_SCALAR_KEYS = {
    "sequencing",
    "Start_Times",
    "room_assignment",
    "start_times"
}


TOTAL_RUNS = (
    len(MODELS)
    *
    len(LAMBDA_LIST)
)


all_models_results = []

completed_runs = 0

experiment_start = time.time()


print("\n" + "=" * 70)
print("STARTING PRODUCTION MILP EXPERIMENT")
print("=" * 70)

print(
    f"Total planned runs: {TOTAL_RUNS}"
)

print(
    "Checkpointing after every completed attempt."
)


# ============================================================
# MODEL LOOP
# ============================================================

for model_name in MODELS:

    print("\n\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)


    # Already verified in Cell 2.

    df_day = (
        verified_model_data[
            model_name
        ]
        .copy()
    )


    model_records = []


    # ========================================================
    # LAMBDA LOOP
    # ========================================================

    for lam in LAMBDA_LIST:

        completed_runs += 1

        lam = float(lam)

        print("\n" + "-" * 70)

        print(
            f"Run {completed_runs}/{TOTAL_RUNS}"
            f" | Model = {model_name}"
            f" | Lambda = {lam:.1f}"
        )

        print("-" * 70)


        run_start = time.time()


        try:

            result = solve_or_milp(

                df_day=df_day,

                lam=lam,

                n_rooms=N_ROOMS,

                room_capacity=ROOM_CAPACITY,

                turnover=TURNOVER,

                balance_weight=BALANCE_WEIGHT,

                duration_cap=DURATION_CAP,

                time_limit=TIME_LIMIT
            )


            # =================================================
            # NO FEASIBLE INCUMBENT
            # =================================================

            if result is None:

                record = {

                    "Model":
                        model_name,

                    "Lambda":
                        lam,

                    "Solver_Failed":
                        True,

                    "Failure_Type":
                        "No feasible incumbent",

                    "Failure_Message":
                        "",

                    "Run_Wall_Time":
                        time.time() - run_start
                }


                print(
                    "WARNING: solver returned no feasible "
                    "incumbent."
                )


            # =================================================
            # SUCCESSFUL SOLVER RETURN
            # =================================================

            else:

                # Remove schedule-level Python objects from the
                # summary table.

                record = {

                    key: value

                    for key, value
                    in result.items()

                    if key not in NON_SCALAR_KEYS
                }


                # Add experiment metadata.

                record["Model"] = (
                    model_name
                )

                record["Solver_Failed"] = (
                    False
                )

                record["Failure_Type"] = (
                    ""
                )

                record["Failure_Message"] = (
                    ""
                )

                record["Run_Wall_Time"] = (
                    time.time()
                    -
                    run_start
                )


                # ---------------------------------------------
                # Immediate diagnostic output
                # ---------------------------------------------

                print(
                    f"Objective       : "
                    f"{record['Objective']:.4f}"
                )

                print(
                    f"Gap             : "
                    f"{record['Gap']:.4%}"
                )

                print(
                    f"Solve time      : "
                    f"{record['Solve_Time']:.2f} s"
                )

                print(
                    f"Nodes           : "
                    f"{record['Nodes']:.0f}"
                )

                print(
                    f"Status          : "
                    f"{record['Status']}"
                )

                print(
                    f"Reached 1% gap  : "
                    f"{record['Reached_1pct_Gap']}"
                )

                print(
                    f"Hit time limit  : "
                    f"{record['Hit_Time_Limit']}"
                )

                print(
                    f"N capped        : "
                    f"{record['N_Capped']}"
                )


        # =====================================================
        # PYTHON / GUROBI EXCEPTION
        # =====================================================

        except Exception as exc:

            record = {

                "Model":
                    model_name,

                "Lambda":
                    lam,

                "Solver_Failed":
                    True,

                "Failure_Type":
                    type(exc).__name__,

                "Failure_Message":
                    str(exc),

                "Run_Wall_Time":
                    time.time() - run_start
            }


            print(
                f"ERROR: {type(exc).__name__}: {exc}"
            )


        # =====================================================
        # STORE RESULT
        # =====================================================

        model_records.append(
            record
        )

        all_models_results.append(
            record
        )


        # =====================================================
        # CHECKPOINT AFTER EVERY RUN
        # =====================================================

        checkpoint_df = pd.DataFrame(
            all_models_results
        )

        checkpoint_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )


        # =====================================================
        # PROGRESS INFORMATION
        # =====================================================

        elapsed = (
            time.time()
            -
            experiment_start
        )

        average_per_run = (
            elapsed
            /
            completed_runs
        )

        remaining_runs = (
            TOTAL_RUNS
            -
            completed_runs
        )

        estimated_remaining = (
            average_per_run
            *
            remaining_runs
        )


        print(
            f"Checkpoint saved : "
            f"{CHECKPOINT_PATH}"
        )

        print(
            f"Elapsed          : "
            f"{elapsed / 60:.1f} min"
        )

        print(
            f"Estimated remain : "
            f"{estimated_remaining / 60:.1f} min"
        )


    # ========================================================
    # EXPORT PER-MODEL RESULTS
    # ========================================================

    model_df = pd.DataFrame(
        model_records
    )

    model_output = os.path.join(
        RESULTS_DIR,
        f"{model_name}_summary.csv"
    )

    model_df.to_csv(
        model_output,
        index=False
    )


    print("\n" + "-" * 70)

    print(
        f"{model_name} results saved -> "
        f"{model_output}"
    )


# ============================================================
# INITIAL MASTER EXPORT
# ============================================================

master_df = pd.DataFrame(
    all_models_results
)

master_df.to_csv(
    MASTER_PATH,
    index=False
)


total_elapsed = (
    time.time()
    -
    experiment_start
)


print("\n" + "=" * 70)
print("PRODUCTION LOOP FINISHED")
print("=" * 70)

print(
    f"Recorded rows      : {len(master_df)}"
)

print(
    f"Total elapsed time : "
    f"{total_elapsed / 60:.2f} min"
)

print(
    f"Master table       : {MASTER_PATH}"
)

In [ ]:
# ============================================================
# FINAL EXPERIMENT INTEGRITY CHECK
# + SUPERVISOR-REQUIRED SOLVER SUMMARY
# ============================================================


EXPECTED_RUNS = (
    len(MODELS)
    *
    len(LAMBDA_LIST)
)


print("\n" + "=" * 70)
print("FINAL EXPERIMENT INTEGRITY CHECK")
print("=" * 70)


# ============================================================
# 1. MASTER ROW COUNT
# ============================================================

assert len(master_df) == EXPECTED_RUNS, (
    f"Expected {EXPECTED_RUNS} experiment rows, "
    f"found {len(master_df)}."
)


# ============================================================
# 2. MODEL × LAMBDA UNIQUENESS
# ============================================================

duplicate_runs = (
    master_df
    .duplicated(
        subset=[
            "Model",
            "Lambda"
        ]
    )
)


assert not duplicate_runs.any(), (
    "Duplicate Model × Lambda runs detected."
)


# ============================================================
# 3. CHECK EACH MODEL HAS ALL 11 LAMBDA VALUES
# ============================================================

expected_lambdas = set(
    float(x)
    for x in LAMBDA_LIST
)


for model_name in MODELS:

    model_rows = master_df[
        master_df["Model"]
        ==
        model_name
    ]

    assert len(model_rows) == len(LAMBDA_LIST), (
        f"{model_name}: expected "
        f"{len(LAMBDA_LIST)} rows, "
        f"found {len(model_rows)}."
    )


    actual_lambdas = set(
        model_rows["Lambda"]
        .astype(float)
        .tolist()
    )


    assert actual_lambdas == expected_lambdas, (
        f"{model_name}: lambda grid mismatch.\n"
        f"Expected: {sorted(expected_lambdas)}\n"
        f"Actual:   {sorted(actual_lambdas)}"
    )


# ============================================================
# 4. CHECK SOLVER FAILURES
# ============================================================

failed_mask = (
    master_df["Solver_Failed"]
    .fillna(True)
    .astype(bool)
)

n_failed = int(
    failed_mask.sum()
)


print(
    f"Expected runs       : {EXPECTED_RUNS}"
)

print(
    f"Recorded runs       : {len(master_df)}"
)

print(
    f"Solver failures     : {n_failed}"
)


if n_failed > 0:

    print("\nFAILED RUNS:")

    print(
        master_df.loc[
            failed_mask,
            [
                "Model",
                "Lambda",
                "Failure_Type",
                "Failure_Message"
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 5. SUCCESSFUL SOLVER RUNS
# ============================================================

successful_df = (
    master_df.loc[
        ~failed_mask
    ]
    .copy()
)


if len(successful_df) == 0:

    raise RuntimeError(
        "No successful MILP runs available "
        "for solver-performance analysis."
    )


# ============================================================
# 6. VERIFY REQUIRED SOLVER DIAGNOSTICS
# ============================================================

required_solver_fields = {

    "Solve_Time",
    "Gap",
    "Status",
    "Reached_1pct_Gap",
    "Hit_Time_Limit",
    "Nodes",
    "Variables",
    "Constraints"
}


missing_solver_fields = (
    required_solver_fields
    -
    set(successful_df.columns)
)


assert len(missing_solver_fields) == 0, (
    "Missing required solver diagnostics: "
    f"{missing_solver_fields}"
)


# ============================================================
# 7. OVERALL SOLVER PERFORMANCE
# ============================================================

overall_summary = {

    "Scope":
        "Overall",

    "Runs":
        len(successful_df),

    "Mean_Solve_Time_s":
        successful_df[
            "Solve_Time"
        ].mean(),

    "Median_Solve_Time_s":
        successful_df[
            "Solve_Time"
        ].median(),

    "Max_Solve_Time_s":
        successful_df[
            "Solve_Time"
        ].max(),

    "Mean_Final_MIP_Gap":
        successful_df[
            "Gap"
        ].mean(),

    "Median_Final_MIP_Gap":
        successful_df[
            "Gap"
        ].median(),

    "Max_Final_MIP_Gap":
        successful_df[
            "Gap"
        ].max(),

    "Reached_1pct_Count":
        int(
            successful_df[
                "Reached_1pct_Gap"
            ]
            .astype(bool)
            .sum()
        ),

    "Reached_1pct_Percent":
        100.0
        *
        successful_df[
            "Reached_1pct_Gap"
        ]
        .astype(bool)
        .mean(),

    "Time_Limit_Count":
        int(
            successful_df[
                "Hit_Time_Limit"
            ]
            .astype(bool)
            .sum()
        ),

    "Time_Limit_Percent":
        100.0
        *
        successful_df[
            "Hit_Time_Limit"
        ]
        .astype(bool)
        .mean(),

    "Mean_Nodes":
        successful_df[
            "Nodes"
        ].mean(),

    "Median_Nodes":
        successful_df[
            "Nodes"
        ].median()
}


# ============================================================
# 8. MODEL-SPECIFIC SOLVER PERFORMANCE
# ============================================================

solver_summary_records = [
    overall_summary
]


for model_name in MODELS:

    model_success = successful_df[
        successful_df["Model"]
        ==
        model_name
    ]


    if len(model_success) == 0:
        continue


    model_summary = {

        "Scope":
            model_name,

        "Runs":
            len(model_success),

        "Mean_Solve_Time_s":
            model_success[
                "Solve_Time"
            ].mean(),

        "Median_Solve_Time_s":
            model_success[
                "Solve_Time"
            ].median(),

        "Max_Solve_Time_s":
            model_success[
                "Solve_Time"
            ].max(),

        "Mean_Final_MIP_Gap":
            model_success[
                "Gap"
            ].mean(),

        "Median_Final_MIP_Gap":
            model_success[
                "Gap"
            ].median(),

        "Max_Final_MIP_Gap":
            model_success[
                "Gap"
            ].max(),

        "Reached_1pct_Count":
            int(
                model_success[
                    "Reached_1pct_Gap"
                ]
                .astype(bool)
                .sum()
            ),

        "Reached_1pct_Percent":
            100.0
            *
            model_success[
                "Reached_1pct_Gap"
            ]
            .astype(bool)
            .mean(),

        "Time_Limit_Count":
            int(
                model_success[
                    "Hit_Time_Limit"
                ]
                .astype(bool)
                .sum()
            ),

        "Time_Limit_Percent":
            100.0
            *
            model_success[
                "Hit_Time_Limit"
            ]
            .astype(bool)
            .mean(),

        "Mean_Nodes":
            model_success[
                "Nodes"
            ].mean(),

        "Median_Nodes":
            model_success[
                "Nodes"
            ].median()
    }


    solver_summary_records.append(
        model_summary
    )


solver_summary_df = pd.DataFrame(
    solver_summary_records
)


# ============================================================
# 9. SAVE SOLVER SUMMARY
# ============================================================

solver_summary_df.to_csv(
    SOLVER_SUMMARY_PATH,
    index=False
)


# ============================================================
# 10. SORT AND RE-SAVE FINAL MASTER TABLE
# ============================================================

model_order = {
    "LightGBM": 0,
    "RF": 1,
    "XGBoost": 2
}


master_df["_Model_Order"] = (
    master_df["Model"]
    .map(model_order)
)


master_df = (
    master_df
    .sort_values(
        [
            "_Model_Order",
            "Lambda"
        ]
    )
    .drop(
        columns="_Model_Order"
    )
    .reset_index(
        drop=True
    )
)


master_df.to_csv(
    MASTER_PATH,
    index=False
)


# ============================================================
# 11. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("FINAL INTEGRITY CHECK: PASS")
print("=" * 70)

print(
    f"33 planned experiment rows present: "
    f"{len(master_df) == EXPECTED_RUNS}"
)

print(
    f"Solver failures: {n_failed}"
)

print(
    f"Master results saved -> "
    f"{MASTER_PATH}"
)

print(
    f"Solver summary saved -> "
    f"{SOLVER_SUMMARY_PATH}"
)


print("\n" + "=" * 70)
print("SUPERVISOR-REQUIRED SOLVER PERFORMANCE")
print("=" * 70)

print(
    solver_summary_df.to_string(
        index=False
    )
)


print("\n" + "=" * 70)
print("MODEL × LAMBDA RUN STATUS")
print("=" * 70)

display_columns = [
    "Model",
    "Lambda",
    "Objective",
    "Gap",
    "Solve_Time",
    "Nodes",
    "Reached_1pct_Gap",
    "Hit_Time_Limit",
    "N_Capped"
]


available_display_columns = [
    column
    for column in display_columns
    if column in master_df.columns
]


print(
    master_df[
        available_display_columns
    ].to_string(
        index=False
    )
)